# V13 레시피 미세 조정

## V13 (Sensitive Layer Protection) 기반 하이퍼파라미터 튜닝

V13의 아키텍처(L0+L29 FP16 보호)를 유지하면서 다음 파라미터를 변경하여 PerfNorm 향상 탐색:

| 실험 | dampening_frac | samples | seq_len | seed | 설명 |
|------|---------------|---------|---------|------|------|
| A | 0.001 | 256 | 512 | default | V13 원본 (기준선) |
| B | **0.0008** | 256 | 512 | default | dampening 감소 |
| C | **0.0005** | 256 | 512 | default | dampening 더 감소 |
| D | 0.001 | **128** | 512 | default | 샘플 수 감소 (GPTQ 논문 기준) |
| E | 0.001 | 256 | 512 | **42** | 캘리브레이션 데이터 셔플 |
| F | 0.001 | 256 | 512 | **123** | 다른 셔플 seed |

---

# 1. Import

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\u26a0\ufe0f CPU 모드로 실행됩니다")
print("\n\u2705 Import 완료!")

# 2. 실험 선택

아래 `EXPERIMENT` 값을 변경하여 원하는 실험을 선택하세요.

In [ ]:
# ============================================================================
# ⭐ 실험 선택 (A~F 중 하나를 선택)
# ============================================================================
EXPERIMENT = "B"  # <<< 이 값만 변경하세요

# ============================================================================
# 실험별 파라미터 정의
# ============================================================================
EXPERIMENTS = {
    "A": {  # V13 원본 (기준선)
        "dampening_frac": 0.001,
        "num_samples": 256,
        "max_seq_len": 512,
        "shuffle_seed": None,
        "desc": "V13 원본 (기준선)",
    },
    "B": {  # dampening 감소
        "dampening_frac": 0.0008,
        "num_samples": 256,
        "max_seq_len": 512,
        "shuffle_seed": None,
        "desc": "dampening 0.0008",
    },
    "C": {  # dampening 더 감소
        "dampening_frac": 0.0005,
        "num_samples": 256,
        "max_seq_len": 512,
        "shuffle_seed": None,
        "desc": "dampening 0.0005",
    },
    "D": {  # 샘플 수 감소 (GPTQ 논문 128개)
        "dampening_frac": 0.001,
        "num_samples": 128,
        "max_seq_len": 512,
        "shuffle_seed": None,
        "desc": "128 samples (GPTQ 논문)",
    },
    "E": {  # 셔플 seed 42
        "dampening_frac": 0.001,
        "num_samples": 256,
        "max_seq_len": 512,
        "shuffle_seed": 42,
        "desc": "shuffle seed=42",
    },
    "F": {  # 셔플 seed 123
        "dampening_frac": 0.001,
        "num_samples": 256,
        "max_seq_len": 512,
        "shuffle_seed": 123,
        "desc": "shuffle seed=123",
    },
}

exp = EXPERIMENTS[EXPERIMENT]

# ============================================================================
# 공통 설정 (변경 금지)
# ============================================================================
MODEL_ID = "./open/base_model"
OUT_DIR = f"./model_exp{EXPERIMENT}"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"
SCHEME = "W4A16"
TARGETS = ["Linear"]
BLOCK_SIZE = 128          # Marlin 호환 필수
ACTORDER = "weight"       # 최적 설정 고정
IGNORE = [
    "embed_tokens", "lm_head",
    "re:model\\.layers\\.0\\..*",
    "re:model\\.layers\\.29\\..*",
]
ORIGINAL_MODEL_SIZE_GB = 2.56

# 실험별 파라미터 적용
DAMPENING_FRAC = exp["dampening_frac"]
NUM_CALIBRATION_SAMPLES = exp["num_samples"]
MAX_SEQUENCE_LENGTH = exp["max_seq_len"]
SHUFFLE_SEED = exp["shuffle_seed"]

print("=" * 60)
print(f"실험 {EXPERIMENT}: {exp['desc']}")
print("=" * 60)
print(f"OUT_DIR: {OUT_DIR}")
print(f"DAMPENING_FRAC: {DAMPENING_FRAC}")
print(f"NUM_SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"MAX_SEQ_LEN: {MAX_SEQUENCE_LENGTH}")
print(f"SHUFFLE_SEED: {SHUFFLE_SEED}")
print(f"BLOCK_SIZE: {BLOCK_SIZE} (Marlin 호환)")
print(f"ACTORDER: {ACTORDER}")
print(f"IGNORE: {IGNORE}")
print("=" * 60)

# 3. 모델 로드

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print("[INFO] 모델/토크나이저 로드 완료")

# 4. 데이터셋 로드

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

if SHUFFLE_SEED is not None:
    # 전체 데이터 로드 후 셔플하여 다른 샘플 선택
    ds = load_dataset(
        DATASET_ID,
        split=f"{DATASET_SPLIT}[:1000]",  # 충분히 로드
    )
    ds = ds.shuffle(seed=SHUFFLE_SEED)
    ds = ds.select(range(NUM_CALIBRATION_SAMPLES))
    print(f"[INFO] 셔플 적용 (seed={SHUFFLE_SEED})")
else:
    ds = load_dataset(
        DATASET_ID,
        split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
    )

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")
print("[INFO] 데이터 전처리 완료")

# 5. GPTQ 양자화

In [ ]:
print(f"[INFO] 실험 {EXPERIMENT} GPTQ 양자화 시작")
print(f"  - dampening_frac: {DAMPENING_FRAC}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - shuffle_seed: {SHUFFLE_SEED}")

if torch.cuda.is_available():
    print("\n\U0001f680 GPU 모드\n")
else:
    print("\n\u23f3 CPU 모드: 오래 걸릴 수 있음\n")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print(f"\n[INFO] 실험 {EXPERIMENT} 양자화 완료!")

# 6. 모델 저장

In [ ]:
print(f"[INFO] 모델 저장: {OUT_DIR}")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print(f"실험 {EXPERIMENT} 모델 크기")
print("=" * 60)
print(f"  원본 모델:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  양자화 모델:   {quantized_size_gb:.2f} GB")
print(f"  압축률:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print("=" * 60)

# 7. 제출 파일 생성

In [ ]:
# submit 디렉토리에 저장
submit_dir = "./submit"
os.makedirs(submit_dir, exist_ok=True)

zip_name = f"submit_exp{EXPERIMENT}"
zip_path = os.path.join(submit_dir, zip_name)

print(f"[INFO] {zip_name}.zip 생성 중...")

if os.path.exists(f"{zip_path}.zip"):
    os.remove(f"{zip_path}.zip")

shutil.make_archive(
    base_name=zip_path,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_path}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_path}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("\u2705 용량 제한 충족 (\u2264 10GB)")
else:
    print("\u274c 용량 초과!")

# 8. 제출용 submit.zip 생성 (선택)

이 실험의 모델을 최종 제출용으로 사용하려면 아래 셀을 실행하세요.

In [ ]:
# 최종 제출용: model/ 디렉토리를 이 실험 결과로 교체 + submit.zip 생성
MAKE_FINAL = False  # True로 변경하면 실행

if MAKE_FINAL:
    # model/ 디렉토리 교체
    final_model_dir = "./model"
    if os.path.exists(final_model_dir):
        shutil.rmtree(final_model_dir)
    shutil.copytree(OUT_DIR, final_model_dir)
    print(f"[INFO] {OUT_DIR} → {final_model_dir} 복사 완료")

    # submit.zip 생성
    if os.path.exists("submit.zip"):
        os.remove("submit.zip")
    shutil.make_archive(
        base_name="submit",
        format="zip",
        root_dir=".",
        base_dir="model",
    )
    final_zip_size = os.path.getsize("submit.zip") / 1e9
    print(f"[INFO] submit.zip 생성 완료 ({final_zip_size:.2f} GB)")
    print(f"\n\u2705 실험 {EXPERIMENT} 모델로 최종 제출 준비 완료!")
else:
    print("[INFO] MAKE_FINAL=False → 건너뜀")
    print("[INFO] 최종 제출하려면 MAKE_FINAL=True로 변경 후 재실행")

---

# 실험 결과 기록

| 실험 | dampening | samples | seed | 모델 크기 | zip 크기 | PerfNorm | SpeedNorm | Score |
|------|-----------|---------|------|----------|---------|----------|-----------|-------|
| A (V13) | 0.001 | 256 | - | 1.42 GB | 0.88 GB | ? | ? | 최고점 |
| B | 0.0008 | 256 | - | | | | | |
| C | 0.0005 | 256 | - | | | | | |
| D | 0.001 | 128 | - | | | | | |
| E | 0.001 | 256 | 42 | | | | | |
| F | 0.001 | 256 | 123 | | | | | |